In [11]:
!pip install groq python-dotenv

In [12]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

print("API Key Loaded:", bool(GROQ_API_KEY))

API Key Loaded: True


In [13]:
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)

MODEL_NAME = "llama-3.1-8b-instant"

In [14]:
MODEL_CONFIG = {
    "technical": {
        "system_prompt": """You are a highly skilled technical support engineer.
Provide precise, code-focused, and practical debugging solutions.
Be concise and accurate."""
    },
    "billing": {
        "system_prompt": """You are a helpful billing support specialist.
Be empathetic, polite, and explain policies clearly.
Help resolve payment and refund issues."""
    },
    "general": {
        "system_prompt": """You are a friendly customer support assistant.
Handle general queries and casual conversation."""
    }
}

In [15]:
def route_prompt(user_input):
    routing_prompt = f"""
Classify the following user query into one of these categories:
[technical, billing, general]

Return ONLY the category name.

Query: {user_input}
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": routing_prompt}],
        temperature=0
    )

    category = response.choices[0].message.content.strip().lower()
    return category

In [16]:
def process_request(user_input):
    category = route_prompt(user_input)

    # Safety fallback
    if category not in MODEL_CONFIG:
        category = "general"

    system_prompt = MODEL_CONFIG[category]["system_prompt"]

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ],
        temperature=0.7
    )

    return {
        "category": category,
        "response": response.choices[0].message.content
    }

In [17]:
queries = [
    "My Python script throws an IndexError on line 5",
    "I was charged twice for my subscription",
    "Hello! What can you do?"
]

for q in queries:
    result = process_request(q)
    print("\n============================")
    print("Query:", q)
    print("Category:", result["category"])
    print("Response:", result["response"])


Query: My Python script throws an IndexError on line 5
Category: technical
Response: To help you debug this issue, I'll need more information. However, I can provide a general approach to troubleshoot `IndexError` in Python.

**General Steps:**

1. **Provide the script code**: Share the Python script where the error occurs, including the relevant lines of code.
2. **Share the error message**: Include the exact error message you see when running the script.
3. **Describe the context**: Tell me how the script is being executed, including any relevant input data or environment variables.

Assuming you've provided the necessary information, I'll guide you through the process of debugging the `IndexError`.

**Common Causes:**

1. **Index out of range**: You're trying to access an index that doesn't exist in a list, tuple, or string.
2. **List/tuple/string access**: You're trying to access an index that's not within the valid range.

**Example Debugging Session:**

If you have the following

In [18]:
MODEL_CONFIG["tool"] = {
    "system_prompt": "You are a tool-using assistant."
}